## Behaviour steering knowledge

This notebook gives nbskill a small, practical memory. A memory item is a regex plus a note: agents can store a repeated good or bad practice once, and later style checks can warn when notebook code matches it.

The lookup is intentionally deterministic. nbskill does not try to infer intent from prose at warning time; it scans the exact notebook or cell being inspected and reports matching rules.

### Production contract

Behaviour steering knowledge is experimental and CLI-first. It must store explicit regex rules, validate regexes before writing, keep default rules inspectable, and surface matches as style diagnostics without changing notebooks.


In [ ]:
#| default_exp knowledge

### Imports

The knowledge file is plain JSON. Notebook scanning only needs notebook reads, code-cell source text, and regex matching.

In [ ]:
#| export
import ast, glob, hashlib, importlib.util, json, math, os, re, shutil, subprocess, tempfile, time

try:
    import tomllib
except ImportError:
    tomllib = None

from pathlib import Path

from fastcore.nbio import read_nb

from nbskill.foundation import cell_source

### Built-in examples

nbskill ships a few default behaviours so the feature works before a user has saved anything. User memory is merged on top of these defaults, and defaults can be disabled with `NBSKILL_KNOWLEDGE_DEFAULTS=0`.

In [ ]:
#| export
_DEFAULT_BEHAVIOURS = [
    {
        "id": "avoid-aliased-function-imports",
        "regex": r"from\s+\S+\s+import\s+\w+\s+as\s+\w+",
        "note": "Do not import functions by alias unless it is needed to avoid a real conflict.",
        "source": "default",
    },
    {
        "id": "avoid-wildcard-imports",
        "regex": r"from\s+\S+\s+import\s+\*",
        "note": "Avoid wildcard imports; import the names used by the cell.",
        "source": "default",
    },
    {
        "id": "avoid-shell-true",
        "regex": r"subprocess\.(run|Popen|call|check_call|check_output)\([^)]*shell\s*=\s*True",
        "note": "Avoid shell=True unless shell behavior is explicitly required.",
        "source": "default",
    },
]


In [ ]:
#| export
_DEFAULT_BEHAVIOURS += [
    {
        "id": "avoid-eval-exec",
        "regex": r"\b(eval|exec)\s*\(",
        "note": "Avoid eval/exec unless dynamic execution is the actual goal.",
        "source": "default",
    },
    {
        "id": "avoid-destructive-git",
        "regex": r"git\s+(reset\s+--hard|clean\s+-fd|checkout\s+--)",
        "note": "Do not run destructive git cleanup commands unless the user explicitly asked for them.",
        "source": "default",
    },
]


In [ ]:
#| export
def default_knowledge():
    "Return built-in behaviour steering examples."
    return {"version": 1, "behaviours": [dict(item) for item in _DEFAULT_BEHAVIOURS]}


In [ ]:
defaults = default_knowledge()["behaviours"]
print([item["id"] for item in defaults[:2]])
assert any(item["id"] == "avoid-aliased-function-imports" for item in defaults)

['avoid-aliased-function-imports', 'avoid-wildcard-imports']


### JSON memory

User-added rules live in one JSON file. The default location is `~/.nbskill-knowledge.json`, with an environment override for tests or project-specific memory.

In [ ]:
#| export
def knowledge_path(path=None):
    "Return the JSON file used for behaviour steering memory."
    default = Path.home() / ".nbskill-knowledge.json"
    return Path(path or os.environ.get("NBSKILL_KNOWLEDGE_PATH", default)).expanduser()

In [ ]:
#| export
def _empty_knowledge():
    return {"version": 1, "behaviours": []}

In [ ]:
#| export
def _defaults_enabled():
    return os.environ.get("NBSKILL_KNOWLEDGE_DEFAULTS", "1").lower() not in {"0", "false", "no"}

In [ ]:
#| export
def _merge_behaviours(*groups):
    seen, behaviours = set(), []
    for group in groups:
        for item in group:
            key = item.get("id") or item.get("regex")
            if key in seen: continue
            seen.add(key)
            behaviours.append(dict(item))
    return behaviours

In [ ]:
#| export
def _load_user_knowledge(path=None):
    pth = knowledge_path(path)
    try: data = json.loads(pth.read_text(encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError, OSError): data = _empty_knowledge()
    data.setdefault("version", 1)
    data.setdefault("behaviours", [])
    return data

In [ ]:
#| export
def load_knowledge(path=None, include_defaults=True):
    "Load behaviour steering rules from defaults and the JSON memory file."
    data = _load_user_knowledge(path)
    if include_defaults and _defaults_enabled():
        data["behaviours"] = _merge_behaviours(default_knowledge()["behaviours"], data.get("behaviours", []))
    return data

In [ ]:
from nbskill.foundation import demo_path, remove_demo_path

memory_path = demo_path("knowledge-empty.json")
try:
    loaded = load_knowledge(memory_path)
    print(loaded["behaviours"][0]["id"])
    assert loaded["behaviours"]
    assert _load_user_knowledge(memory_path)["behaviours"] == []
finally:
    remove_demo_path(memory_path)

avoid-aliased-function-imports


### Storing and retrieving rules

`store_knowledge` is the main write path: the agent provides the regex it created and the note that should be shown when it matches. `get_knowledge` can inspect all rules or filter by regex against the stored regex and note text.

In [ ]:
#| export
def _write_knowledge(data, path=None):
    pth = knowledge_path(path)
    pth.parent.mkdir(parents=True, exist_ok=True)
    pth.write_text(json.dumps(data, indent=2, sort_keys=True), encoding="utf-8")
    return pth

In [ ]:
#| export
def _check_regex(pattern):
    re.compile(pattern)
    return pattern

In [ ]:
#| export
def store_knowledge(
    apply_regex: str,  # Regex to apply to notebook code cells
    note: str,  # Behaviour note shown when the regex matches
    path: str | None = None,  # Override JSON memory path
):
    "Store or update one behaviour steering regex and note."
    _check_regex(apply_regex)
    data = _load_user_knowledge(path)
    now = time.time()
    item = {"regex": apply_regex, "note": note, "updated_ts": now}
    for idx, existing in enumerate(data["behaviours"]):
        if existing.get("regex") == apply_regex:
            item["created_ts"] = existing.get("created_ts", now)
            data["behaviours"][idx] = {**existing, **item}
            break
    else:
        item["created_ts"] = now
        data["behaviours"].append(item)
    pth = _write_knowledge(data, path)
    return {"path": str(pth), "stored": item, "count": len(data["behaviours"])}


In [ ]:
#| export
def add_behaviour_steering(
    regex: str,  # Regex the agent created from a behaviour note
    path: str | None = None,  # Override JSON memory path
):
    "Add a behaviour steering regex with a generic note."
    return store_knowledge(regex, f"Behaviour steering matched: {regex}", path=path)

In [ ]:
#| export
def get_knowledge(
    regex: str | None = None,  # Optional regex for filtering stored regexes and notes
    path: str | None = None,  # Override JSON memory path
    include_defaults: bool = True,  # Include built-in example behaviours
):
    "Return stored behaviour steering rules, optionally filtered by regex."
    data = load_knowledge(path, include_defaults=include_defaults)
    items = data.get("behaviours", [])
    if regex:
        query_re = re.compile(_check_regex(regex))
        items = [item for item in items if query_re.search(item.get("regex", "")) or query_re.search(item.get("note", ""))]
    return {"path": str(knowledge_path(path)), "count": len(items), "behaviours": items}


In [ ]:
memory_path = demo_path("knowledge-store.json")
try:
    stored = store_knowledge(r"from\s+\S+\s+import\s+\w+\s+as\s+\w+", "Do not import functions by alias unless needed.", path=str(memory_path))
    found = get_knowledge("alias", path=str(memory_path))
    print(found["behaviours"][-1]["note"])
    assert stored["count"] == 1
    assert found["count"] >= 1
finally:
    remove_demo_path(memory_path)

{
  "count": 1,
  "path": "nbs/data/knowledge-store.json",
  "stored": {
    "created_ts": 1779270447.670509,
    "note": "Do not import functions by alias unless needed.",
    "regex": "from\\s+\\S+\\s+import\\s+\\w+\\s+as\\s+\\w+",
    "updated_ts": 1779270447.670509
  }
}
{
  "behaviours": [
    {
      "id": "avoid-aliased-function-imports",
      "note": "Do not import functions by alias unless it is needed to avoid a real conflict.",
      "regex": "from\\s+\\S+\\s+import\\s+\\w+\\s+as\\s+\\w+",
      "source": "default"
    },
    {
      "created_ts": 1779270447.670509,
      "note": "Do not import functions by alias unless needed.",
      "regex": "from\\s+\\S+\\s+import\\s+\\w+\\s+as\\s+\\w+",
      "updated_ts": 1779270447.670509
    }
  ],
  "count": 2,
  "path": "nbs/data/knowledge-store.json"
}
Do not import functions by alias unless needed.


### Applying memory to notebooks

Style checks use the same matcher as MCP warnings. A rule is only useful if it points at a concrete notebook cell and line, so diagnostics include the path, cell id, line, note, regex, and matched text.

In [ ]:
#| export
def _is_notebook_path(path):
    path = Path(path)
    return path.suffix == ".ipynb" and ".ipynb_checkpoints" not in path.parts

In [ ]:
#| export
def _notebook_paths(path="."):
    raw = str(path)
    pth = Path(raw).expanduser()
    if any(char in raw for char in "*?[]"): candidates = [Path(item) for item in glob.glob(raw, recursive=True)]
    elif pth.is_dir(): candidates = list(pth.rglob("*.ipynb"))
    elif pth.is_file() and pth.suffix == ".ipynb": candidates = [pth]
    else: candidates = []
    return sorted({candidate for candidate in candidates if _is_notebook_path(candidate)})

In [ ]:
#| export
def _line_for_match(source, match):
    return source.count("\n", 0, match.start()) + 1

In [ ]:
#| export
def _knowledge_problem(nb_path, cell, line, item, match):
    note = item.get("note", "")
    return {
        "code": "knowledge-warning",
        "path": str(nb_path),
        "severity": "warning",
        "source": "nbskill-knowledge",
        "detail": "stored behaviour steering rule matched",
        "cell_id": getattr(cell, "id", ""),
        "line": line,
        "regex": item.get("regex", ""),
        "note": note,
        "match": match.group(0).splitlines()[0][:160],
    }

In [ ]:
#| export
def knowledge_style_problems(path=".", knowledge_path_override=None):
    "Return style diagnostics for stored behaviour steering regex matches."
    rules = load_knowledge(knowledge_path_override).get("behaviours", [])
    compiled = [(item, re.compile(item["regex"], re.MULTILINE)) for item in rules if item.get("regex")]
    if not compiled: return []
    problems = []
    for nb_path in _notebook_paths(path):
        try: nb = read_nb(nb_path)
        except FileNotFoundError: continue
        for cell in nb.cells:
            if getattr(cell, "cell_type", None) != "code": continue
            source = cell_source(cell)
            for item, pattern in compiled:
                for match in pattern.finditer(source):
                    problems.append(_knowledge_problem(nb_path, cell, _line_for_match(source, match), item, match))
    return problems

In [ ]:
from nbskill.foundation import write_demo_notebook
from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import write_nb as _write_raw_nb

with write_demo_notebook("knowledge-match.json") as memory_path, write_demo_notebook("knowledge-demo.ipynb", reset=False) as demo_nb_path:
    store_knowledge(r"from\s+\S+\s+import\s+\w+\s+as\s+\w+", "Do not import functions by alias unless needed.", path=str(memory_path))
    _write_raw_nb(new_nb([mk_cell("from math import sqrt as square_root\nvalue = square_root(4)")]), demo_nb_path)
    problems = knowledge_style_problems(str(demo_nb_path), knowledge_path_override=str(memory_path))
    print(problems[0]["note"])
    print(len(problems),"problems found")
    assert len(problems)
    assert problems[0]["line"] == 1

{
  "count": 1,
  "path": "nbs/data/knowledge-match.json",
  "stored": {
    "created_ts": 1779270553.4094799,
    "note": "Do not import functions by alias unless needed.",
    "regex": "from\\s+\\S+\\s+import\\s+\\w+\\s+as\\s+\\w+",
    "updated_ts": 1779270553.4094799
  }
}
Do not import functions by alias unless it is needed to avoid a real conflict.
2 problems found


### Reference implementation knowledgebase

The reference knowledgebase stores favourite implementation repositories in one local LanceDB database. It keeps three tables: `reference_repos` for registered sources, `reference_items` for README/module/function/class/method records, and `reference_edges` for direct same-repo caller/callee links.

This feature requires both LanceDB and sentence-transformers. Ingestion writes a full sentence-transformers vector for every indexed item, and queries combine structured filters, LanceDB full-text search, and vector similarity over that searchable vector store. If either dependency or the configured embedding model is unavailable, reference ingestion/querying fails clearly instead of falling back to a weaker embedding.

The public workflow is:

1. `reference_add(...)` registers a local path or GitHub repository.
2. `reference_ingest(...)` clones with `gh repo clone` when possible, extracts Python and notebook symbols, and writes LanceDB rows.
3. `reference_query(...)` returns examples, dependency status, optional branch context, and increments each returned item's `returned_count`.

Reference knowledge examples below use tiny local fixtures. They document the public API without touching the global user database.

Small executable examples appear after the reference API definitions so they can call the exported functions directly.

Small executable examples appear after the reference API definitions so they can call the exported functions directly.

In [ ]:
#| export
def reference_home(path=None):
    "Return the global reference knowledge directory."
    default = Path.home() / ".nbskill" / "reference_knowledge"
    return Path(path or os.environ.get("NBSKILL_REFERENCE_HOME", default)).expanduser()

In [ ]:
#| export
def _reference_table_names():
    return {
        "repos": "reference_repos",
        "items": "reference_items",
        "edges": "reference_edges",
    }

In [ ]:
#| export
def _require_sentence_transformers():
    try:
        import sentence_transformers
        return sentence_transformers
    except Exception as exc:
        raise RuntimeError(f"sentence-transformers is required for reference knowledge: {type(exc).__name__}: {exc}") from exc

def _reference_db(path=None):
    lancedb, reason = _try_lancedb()
    if lancedb is None: raise RuntimeError(f"LanceDB is required for reference knowledge: {reason}")
    _require_sentence_transformers()
    home = reference_home(path)
    home.mkdir(parents=True, exist_ok=True)
    return lancedb.connect(str(_lancedb_path(path)))

In [ ]:
#| export
def _lancedb_path(path=None):
    return reference_home(path) / "lancedb"

def _reference_items_table(path=None):
    return _reference_table(path, "items")

In [ ]:
#| export
_REFERENCE_SYNONYM_GROUPS = [
    {"route", "endpoint", "handler", "url", "page"},
    {"button", "control", "clickable", "action"},
    {"build", "make", "create", "render", "compose"},
    {"website", "web", "html", "fasthtml"},
    {"payment", "billing", "stripe", "checkout"},
    {"attribute", "attributes", "attr", "attrs", "field", "fields"},
    {"argument", "arguments", "arg", "args", "parameter", "parameters", "param", "params"},
    {"constructor", "init", "__init__", "initializer"},
    {"compose", "composition", "pipeline", "pipe", "chain"},
    {"patch", "monkeypatch"},
]

In [ ]:
#| export
def _load_reference_registry(path=None):
    repos = {}
    for row in _lance_rows(path, "repos"):
        name = row.get("name")
        if name: repos[name] = row
    return {"version": 1, "repos": repos}

In [ ]:
#| export
def _write_reference_registry(data, path=None):
    rows = sorted(data.get("repos", {}).values(), key=lambda row: row.get("name", ""))
    _write_lance_table(path, "repos", rows)
    return _lancedb_path(path)

In [ ]:
#| export
def _repo_name_from_source(source):
    text = str(source).rstrip("/")
    name = Path(text).name or text.rsplit("/", 1)[-1]
    return re.sub(r"\.git$", "", name) or "reference"

In [ ]:
#| export
def _norm_dist_name(name):
    return re.sub(r"[-_.]+", "-", str(name or "").lower()).strip("-")

In [ ]:
#| export
def reference_add(
    url: str,  # Repository URL or local repository path
    name: str | None = None,  # Stable registry name; defaults from URL/path
    version: str = "HEAD",  # Git ref to ingest
    package: str | None = None,  # Import/distribution package name
    path: str | None = None,  # Override reference home
):
    "Register or update one reference repository."
    data = _load_reference_registry(path)
    repo_name = name or _repo_name_from_source(url)
    now = time.time()
    existing = data["repos"].get(repo_name, {})
    data["repos"][repo_name] = {
        **existing,
        "name": repo_name,
        "url": url,
        "version": version,
        "package": package,
        "updated_ts": now,
        "created_ts": existing.get("created_ts", now),
    }
    pth = _write_reference_registry(data, path)
    return {"path": str(pth), "repo": data["repos"][repo_name], "count": len(data["repos"])}

In [ ]:
#| export
def _reference_table(path, key):
    db = _reference_db(path)
    name = _reference_table_names()[key]
    try: return db.open_table(name)
    except Exception: return None

def _lance_rows(path, key):
    table = _reference_table(path, key)
    if table is None: return []
    try: return [dict(row) for row in table.to_arrow().to_pylist()]
    except Exception: return []

def _prepare_lance_rows(key, rows):
    if key != "items": return list(rows)
    rows = [dict(row) for row in rows]
    vectors = _embed_reference_texts(row.get("search_text", "") for row in rows)
    backend = _reference_embedding_backend(len(vectors[0]) if vectors else None)
    for row, vector in zip(rows, vectors):
        row["vector"] = vector
        row["embedding_backend"] = backend
        row["returned_count"] = int(row.get("returned_count") or 0)
    return rows

def _write_lance_table(path, key, rows):
    db = _reference_db(path)
    name = _reference_table_names()[key]
    rows = _prepare_lance_rows(key, rows)
    if not rows:
        table = _reference_table(path, key)
        if table is not None: table.delete("name is not null" if key == "repos" else "item_id is not null" if key == "items" else "caller_id is not null")
        return None
    table = db.create_table(name, data=rows, mode="overwrite")
    if key == "items":
        try: table.create_fts_index("search_text", replace=True)
        except Exception: pass
    return table

In [ ]:
#| export
def _reference_storage_note():
    return "Reference knowledge is stored only in LanceDB tables."

In [ ]:
#| export
def _stable_item_id(*parts):
    raw = "\0".join(str(part or "") for part in parts)
    return hashlib.sha1(raw.encode("utf-8")).hexdigest()

In [ ]:
#| export
def _source_excerpt(source, limit=2000):
    text = (source or "").strip()
    return text if len(text) <= limit else text[:limit].rstrip() + "\n..."

In [ ]:
#| export
def _try_lancedb():
    try:
        import lancedb
        return lancedb, None
    except Exception as exc:
        return None, f"lancedb unavailable: {type(exc).__name__}: {exc}"

In [ ]:
#| export
def _all_reference_items(path=None):
    return sorted(_lance_rows(path, "items"), key=lambda row: (row.get("repo", ""), row.get("path", ""), row.get("symbol", "")))

In [ ]:
#| export
def _reference_token_map():
    token_map = {}
    for group in _REFERENCE_SYNONYM_GROUPS:
        expanded = set(group)
        for token in group: token_map[token] = expanded
    return token_map

In [ ]:
#| export
_REFERENCE_SENTENCE_MODEL = os.environ.get("NBSKILL_REFERENCE_SENTENCE_MODEL", "all-MiniLM-L6-v2")
_REFERENCE_SENTENCE_TRANSFORMER = None
_REFERENCE_SENTENCE_NOTE = None

def _sentence_transformer():
    global _REFERENCE_SENTENCE_TRANSFORMER, _REFERENCE_SENTENCE_NOTE
    if _REFERENCE_SENTENCE_TRANSFORMER is not None: return _REFERENCE_SENTENCE_TRANSFORMER
    try:
        from sentence_transformers import SentenceTransformer
    except Exception as exc:
        _REFERENCE_SENTENCE_NOTE = f"sentence-transformers unavailable: {type(exc).__name__}: {exc}"
        raise RuntimeError(_REFERENCE_SENTENCE_NOTE) from exc
    try:
        _REFERENCE_SENTENCE_TRANSFORMER = SentenceTransformer(_REFERENCE_SENTENCE_MODEL, local_files_only=True)
        _REFERENCE_SENTENCE_NOTE = None
        return _REFERENCE_SENTENCE_TRANSFORMER
    except Exception as exc:
        _REFERENCE_SENTENCE_NOTE = f"sentence-transformers model unavailable locally: {type(exc).__name__}: {exc}"
        raise RuntimeError(_REFERENCE_SENTENCE_NOTE) from exc

def _reference_tokens(text):
    token_map = _reference_token_map()
    tokens = []
    for token in re.findall(r"[A-Za-z0-9_]+", str(text).lower()):
        tokens.extend(sorted(token_map.get(token, {token})))
    return tokens

def _reference_vector(text, dims=None):
    model = _sentence_transformer()
    vector = [float(value) for value in model.encode(str(text), normalize_embeddings=True)]
    if dims is not None and len(vector) != dims:
        raise RuntimeError(f"Embedding dimension mismatch: query vector has {len(vector)} dims, index has {dims}; reingest references")
    return vector

def _embed_reference_texts(texts, dims=None):
    texts = list(texts)
    model = _sentence_transformer()
    vectors = [[float(value) for value in row] for row in model.encode(texts, normalize_embeddings=True)]
    if dims is not None and any(len(vector) != dims for vector in vectors):
        raise RuntimeError(f"Embedding dimension mismatch while writing LanceDB rows; expected {dims}")
    return vectors

def _reference_embedding_backend(dims=None):
    vector = _reference_vector("dimension probe")
    if dims is not None and len(vector) != dims:
        raise RuntimeError(f"Embedding dimension mismatch: model has {len(vector)} dims, index has {dims}; reingest references")
    return f"sentence-transformers:{_REFERENCE_SENTENCE_MODEL}"

def _reference_embedding_note():
    try:
        _sentence_transformer()
        return None
    except RuntimeError as exc:
        return str(exc)

In [ ]:
#| export
def _sync_lancedb(path=None):
    rows = _all_reference_items(path)
    table = _reference_table(path, "items")
    if table is not None:
        try: table.create_fts_index("search_text", replace=True)
        except Exception: pass
    return {"ok": True, "backend": "lancedb", "count": len(rows)}

In [ ]:
#| export
def _merge_lancedb_rows(*groups):
    merged = {}
    for source, rows in groups:
        for row in rows or []:
            key = row.get("item_id")
            if not key: continue
            current = merged.setdefault(key, row)
            sources = set(current.get("_search_sources") or [])
            sources.add(source)
            current["_search_sources"] = sorted(sources)
            current["_reference_score"] = max(float(current.get("_reference_score") or 0), _reference_rank_score(row))
    return [row for row in merged.values() if row.get("item_id")]

def _lance_literal(value):
    return "'" + str(value).replace("'", "''") + "'"

def _reference_filter_expr(repos=None, kind=None, package=None, module=None, symbol=None):
    clauses = []
    values = {"kind": kind, "package": package, "module": module, "symbol": symbol}
    for field, value in values.items():
        if value: clauses.append(f"{field} = {_lance_literal(value)}")
    repo_names = _repo_filter(repos)
    if repo_names:
        clauses.append("repo IN (" + ", ".join(_lance_literal(repo) for repo in repo_names) + ")")
    return " AND ".join(clauses)

def _row_matches_reference_filter(row, repos=None, kind=None, package=None, module=None, symbol=None):
    repo_names = set(_repo_filter(repos) or [])
    if repo_names and row.get("repo") not in repo_names: return False
    for field, value in {"kind": kind, "package": package, "module": module, "symbol": symbol}.items():
        if value and row.get(field) != value: return False
    return True

def _reference_rank_score(row, query=""):
    base = float(row.get("_relevance_score") or row.get("_score") or row.get("_reference_score") or 0)
    score = math.log1p(max(base, 0.0))
    if row.get("_distance") is not None:
        try: score += 1.0 / (1.0 + max(float(row["_distance"]), 0.0))
        except (TypeError, ValueError): pass
    tokens = set(_reference_tokens(query))
    symbol = str(row.get("symbol", "") or "").lower().replace("_", " ")
    module = str(row.get("module", "") or "").lower()
    doc = str(row.get("docstring", "") or "").lower()
    sig = str(row.get("signature", "") or "").lower()
    src = str(row.get("source", "") or "").lower()
    symbol_parts = set(re.findall(r"[a-z0-9]+", symbol))
    for token in tokens:
        if token in {symbol, symbol.replace(" ", "_")}: score += 10.0
        elif token in symbol_parts: score += 3.0
        elif token in symbol: score += 1.5
        if token in doc: score += 1.0
        if token in sig: score += 0.6
        if token in module: score += 0.3
        if token in src: score += 0.1
    if tokens and all(token in doc or token in symbol or token in sig for token in tokens): score += 2.0
    if len(symbol.replace(' ', '')) <= 1 and not doc: score -= 8.0
    return score

def _prefer_reference_row(current, candidate):
    if current is None: return candidate
    cur_path, cand_path = current.get("path", ""), candidate.get("path", "")
    if cur_path.startswith("nbs/") and not cand_path.startswith("nbs/"): return candidate
    if candidate.get("_reference_score", 0) > current.get("_reference_score", 0): return candidate
    return current

def _dedupe_reference_rows(rows):
    deduped = {}
    for row in rows:
        source_hash = hashlib.sha1(str(row.get("source", "")).encode("utf-8")).hexdigest()
        key = (row.get("repo"), row.get("kind"), row.get("symbol"), source_hash)
        deduped[key] = _prefer_reference_row(deduped.get(key), row)
    return sorted(deduped.values(), key=lambda row: row.get("_reference_score", 0), reverse=True)

def _search_rows(builder, limit, filter_expr=None):
    try:
        if filter_expr: builder = builder.where(filter_expr, prefilter=True)
    except TypeError:
        if filter_expr: builder = builder.where(filter_expr)
    try: return builder.limit(limit).to_list()
    except Exception: return []

def _lancedb_search(query, top_k=3, repos=None, path=None, kind=None, package=None, module=None, symbol=None):
    table = _reference_table(path, "items")
    if table is None: return [], None, "lancedb"
    limit = max(top_k * 20, 80)
    items = _lance_rows(path, "items")
    dims = next((len(row.get("vector") or []) for row in items if row.get("vector")), None)
    vector = _reference_vector(query, dims=dims)
    text_query = " ".join(_query_tokens(query)) or query
    filter_expr = _reference_filter_expr(repos=repos, kind=kind, package=package, module=module, symbol=symbol)
    try:
        hybrid = _search_rows(table.search(query_type="hybrid").vector(vector).text(text_query), limit, filter_expr)
    except Exception:
        hybrid = []
    vector_rows = _search_rows(table.search(vector).metric("cosine"), limit, filter_expr)
    fts_rows = _search_rows(table.search(text_query, query_type="fts"), limit, filter_expr)
    rows = _merge_lancedb_rows(("hybrid", hybrid), ("vector", vector_rows), ("bm25_fts", fts_rows))
    rows = [row for row in rows if _row_matches_reference_filter(row, repos=repos, kind=kind, package=package, module=module, symbol=symbol)]
    for row in rows: row["_reference_score"] = _reference_rank_score(row, query=query)
    rows = _dedupe_reference_rows(sorted(rows, key=lambda row: row.get("_reference_score", 0), reverse=True))
    note = _reference_embedding_note()
    backend = "lancedb_hybrid_bm25_vector"
    return rows[:top_k], note, backend

In [ ]:
#| export
def _node_name(node):
    if isinstance(node, ast.Name): return node.id
    if isinstance(node, ast.Attribute):
        base = _node_name(node.value)
        return f"{base}.{node.attr}" if base else node.attr
    if isinstance(node, ast.Call): return _node_name(node.func)
    try: return ast.unparse(node)
    except Exception: return ""

In [ ]:
#| export
def _call_names(node):
    names = []
    for child in ast.walk(node):
        if not isinstance(child, ast.Call): continue
        name = _node_name(child.func)
        if name: names.append(name)
    return list(dict.fromkeys(names))

In [ ]:
#| export
def _import_names(tree):
    names = []
    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            names.extend(alias.name.split(".", 1)[0] for alias in node.names)
        elif isinstance(node, ast.ImportFrom) and node.module:
            names.append(node.module.split(".", 1)[0])
    return sorted(set(names))

In [ ]:
#| export
def _decorator_names(node):
    return [name for item in getattr(node, "decorator_list", []) if (name := _node_name(item))]

In [ ]:
#| export
def _signature(node):
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        prefix = "async def" if isinstance(node, ast.AsyncFunctionDef) else "def"
        return f"{prefix} {node.name}({ast.unparse(node.args)})"
    if isinstance(node, ast.ClassDef):
        bases = [ast.unparse(base) for base in node.bases]
        return f"class {node.name}({', '.join(bases)})" if bases else f"class {node.name}"
    return ""

In [ ]:
#| export
def _module_name_from_path(rel_path):
    path = Path(rel_path)
    parts = list(path.with_suffix("").parts)
    if parts and parts[-1] == "__init__": parts = parts[:-1]
    return ".".join(parts)

In [ ]:
#| export
def _row_tags(repo, package, module, kind, symbol, decorators, imports):
    parts = [repo, package, kind, *(module or "").split("."), *(symbol or "").replace(".", " ").split()]
    parts.extend(decorators or [])
    parts.extend(imports or [])
    return sorted({part for part in parts if part})

In [ ]:
#| export
def _item_row(repo, version, package, kind, module, symbol, rel_path, source, **extra):
    search_text = "\n".join(str(extra.get(key, "") or "") for key in ("signature", "docstring"))
    search_text = "\n".join([repo, package or "", kind, module or "", symbol or "", search_text, source or ""])
    item_id = _stable_item_id(repo, version, rel_path, extra.get("cell_id"), kind, symbol, extra.get("start_line"))
    tags = _row_tags(repo, package, module, kind, symbol, extra.get("decorators"), extra.get("imports"))
    return {
        "item_id": item_id, "repo": repo, "version": version, "package": package or "",
        "kind": kind, "module": module or "", "symbol": symbol or "", "path": str(rel_path),
        "cell_id": extra.get("cell_id", ""), "start_line": extra.get("start_line", 1),
        "end_line": extra.get("end_line", 1), "signature": extra.get("signature", ""),
        "docstring": extra.get("docstring", ""), "source": _source_excerpt(source),
        "search_text": search_text, "tags": tags, "calls": extra.get("calls", []),
        "vector": _reference_vector(search_text), "embedding_backend": _reference_embedding_backend(),
        "returned_count": int(extra.get("returned_count") or 0),
    }

In [ ]:
#| export
def _class_block_source(source, node):
    init = next((child for child in node.body if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)) and child.name == "__init__"), None)
    if init is None: return ast.get_source_segment(source, node) or ""
    lines = source.splitlines()
    return "\n".join(lines[node.lineno - 1:getattr(init, "end_lineno", init.lineno)])

def _extract_def_rows(tree, source, repo, version, package, rel_path, module, cell_id=""):
    rows, imports = [], _import_names(tree)
    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            body = ast.get_source_segment(source, node) or ""
            rows.append(_item_row(
                repo, version, package, "function", module, node.name, rel_path, body,
                cell_id=cell_id, start_line=node.lineno, end_line=getattr(node, "end_lineno", node.lineno),
                signature=_signature(node), docstring=ast.get_docstring(node) or "",
                decorators=_decorator_names(node), imports=imports, calls=_call_names(node),
            ))
        elif isinstance(node, ast.ClassDef):
            body = _class_block_source(source, node)
            rows.append(_item_row(
                repo, version, package, "class", module, node.name, rel_path, body,
                cell_id=cell_id, start_line=node.lineno, end_line=getattr(node, "end_lineno", node.lineno),
                signature=_signature(node), docstring=ast.get_docstring(node) or "",
                decorators=_decorator_names(node), imports=imports, calls=_call_names(node),
            ))
            for child in node.body:
                if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)) and child.name != "__init__":
                    method_body = ast.get_source_segment(source, child) or ""
                    rows.append(_item_row(
                        repo, version, package, "method", module, f"{node.name}.{child.name}", rel_path, method_body,
                        cell_id=cell_id, start_line=child.lineno, end_line=getattr(child, "end_lineno", child.lineno),
                        signature=_signature(child), docstring=ast.get_docstring(child) or "",
                        decorators=_decorator_names(child), imports=imports, calls=_call_names(child),
                    ))
    return rows

In [ ]:
#| export
def _extract_python_rows(path, root, repo, version, package):
    rel_path = path.relative_to(root)
    source = path.read_text(encoding="utf-8", errors="replace")
    try: tree = ast.parse(source)
    except SyntaxError: return []
    rows = []
    module = _module_name_from_path(rel_path)
    if doc := ast.get_docstring(tree):
        rows.append(_item_row(repo, version, package, "module", module, module, rel_path, doc, docstring=doc, imports=_import_names(tree)))
    rows.extend(_extract_def_rows(tree, source, repo, version, package, rel_path, module))
    return rows

In [ ]:
#| export
def _extract_notebook_rows(path, root, repo, version, package):
    rel_path = path.relative_to(root)
    try: nb = read_nb(path)
    except Exception: return []
    rows, module = [], _module_name_from_path(rel_path)
    for idx, cell in enumerate(nb.cells):
        source = cell_source(cell)
        cell_id = getattr(cell, "id", "") or str(idx)
        if getattr(cell, "cell_type", None) == "markdown" and source.strip():
            rows.append(_item_row(repo, version, package, "notebook_doc", module, f"{module}#{cell_id}", rel_path, source, cell_id=cell_id))
        if getattr(cell, "cell_type", None) != "code": continue
        try: tree = ast.parse(source)
        except SyntaxError: continue
        rows.extend(_extract_def_rows(tree, source, repo, version, package, rel_path, module, cell_id=cell_id))
    return rows

In [ ]:
#| export
def _pyproject_data(root):
    pyproject = Path(root) / "pyproject.toml"
    if not pyproject.exists() or tomllib is None: return {}
    try:
        return tomllib.loads(pyproject.read_text(encoding="utf-8", errors="replace"))
    except Exception:
        return {}

def _nbdev_config(root):
    data = _pyproject_data(root)
    tool = data.get("tool", {}) if isinstance(data, dict) else {}
    nbdev = tool.get("nbdev", {}) if isinstance(tool, dict) else {}
    if not nbdev: return None
    nbs_path = nbdev.get("nbs_path") or nbdev.get("nbs-path") or "nbs"
    lib_path = nbdev.get("lib_path") or nbdev.get("lib-path") or nbdev.get("lib_name")
    return {"kind": "nbdev", "nbs_path": str(nbs_path), "lib_path": str(lib_path or "")}

def _supported_reference_files(root):
    root = Path(root)
    skip = {".git", ".venv", "__pycache__", ".mypy_cache", ".pytest_cache", ".ipynb_checkpoints"}
    nbdev = _nbdev_config(root)
    if nbdev:
        nbs_root = root / nbdev["nbs_path"]
        if nbs_root.exists():
            for path in nbs_root.rglob("*.ipynb"):
                if not any(part in skip for part in path.parts): yield path
        return
    for path in root.rglob("*"):
        if any(part in skip for part in path.parts): continue
        if path.suffix == ".py" or path.suffix == ".ipynb":
            yield path

In [ ]:
#| export
def _readme_rows(root, repo, version, package):
    rows = []
    for name in ("README.md", "README.rst", "README.txt"):
        path = Path(root) / name
        if path.exists():
            source = path.read_text(encoding="utf-8", errors="replace")
            rows.append(_item_row(repo, version, package, "readme", "", name, path.relative_to(root), source))
    return rows

In [ ]:
#| export
def _run_git(args, cwd):
    proc = subprocess.run(["git", *args], cwd=cwd, text=True, capture_output=True)
    if proc.returncode != 0: raise RuntimeError(proc.stderr.strip() or proc.stdout.strip())
    return proc.stdout.strip()

In [ ]:
#| export
def _github_repo_slug(source):
    text = str(source).strip().removesuffix(".git")
    if re.match(r"^[\w.-]+/[\w.-]+$", text): return text
    match = re.match(r"^(?:https://github\.com/|git@github\.com:)([\w.-]+/[\w.-]+)$", text)
    return match.group(1) if match else None

def _clone_reference_with_gh(source, dest):
    slug = _github_repo_slug(source)
    if not slug or shutil.which("gh") is None: return False
    proc = subprocess.run(["gh", "repo", "clone", slug, str(dest), "--", "--quiet"], text=True, capture_output=True)
    if proc.returncode != 0: return False
    return True

def _checkout_reference(repo, dest):
    source = repo["url"]
    source_path = Path(source).expanduser()
    if source_path.exists() and not (source_path / ".git").exists():
        shutil.copytree(source_path, dest)
    elif not _clone_reference_with_gh(source, dest):
        subprocess.run(["git", "clone", "--quiet", source, str(dest)], check=True)
    version = repo.get("version") or "HEAD"
    if version != "HEAD": _run_git(["checkout", "--quiet", version], dest)
    if (Path(dest) / ".git").exists():
        try: return _run_git(["rev-parse", "HEAD"], dest)
        except RuntimeError: return repo.get("version") or "local"
    return repo.get("version") or "local"

In [ ]:
#| export
def _infer_package(root, repo):
    if repo.get("package"): return repo["package"]
    pyproject = Path(root) / "pyproject.toml"
    if pyproject.exists():
        text = pyproject.read_text(encoding="utf-8", errors="replace")
        if match := re.search(r"(?m)^name\s*=\s*['\"]([^'\"]+)['\"]", text):
            return match.group(1)
    for child in Path(root).iterdir():
        if child.is_dir() and (child / "__init__.py").exists(): return child.name
    return repo["name"].replace("-", "_")

In [ ]:
#| export
def _extract_reference_rows(root, repo, version, package):
    rows = _readme_rows(root, repo["name"], version, package)
    repo_kind = "nbdev" if _nbdev_config(root) else "python"
    for row in rows: row["repo_kind"] = repo_kind
    for file_path in _supported_reference_files(root):
        if file_path.suffix == ".py":
            extracted = _extract_python_rows(file_path, root, repo["name"], version, package)
        elif file_path.suffix == ".ipynb":
            extracted = _extract_notebook_rows(file_path, root, repo["name"], version, package)
        else:
            extracted = []
        for row in extracted: row["repo_kind"] = repo_kind
        rows.extend(extracted)
    return rows

In [ ]:
#| export
def _write_reference_rows(repo_name, version, rows, path=None):
    existing = [row for row in _lance_rows(path, "items") if row.get("repo") != repo_name]
    _write_lance_table(path, "items", [*existing, *rows])
    existing_edges = [row for row in _lance_rows(path, "edges") if row.get("repo") != repo_name]
    _write_lance_table(path, "edges", [*existing_edges, *_reference_edge_rows(repo_name, version, rows)])
    return len(rows)

In [ ]:
#| export
def _symbol_keys(row):
    symbol, module = row.get("symbol", ""), row.get("module", "")
    keys = {symbol, symbol.rsplit(".", 1)[-1]}
    if module and symbol: keys.add(f"{module}.{symbol}")
    return {key for key in keys if key}

In [ ]:
#| export
def _reference_edge_rows(repo_name, version, rows):
    symbol_map = {}
    for row in rows:
        if row.get("kind") not in {"function", "class", "method"}: continue
        for key in _symbol_keys(row): symbol_map.setdefault(key, []).append(row["item_id"])
    edges = []
    seen = set()
    for row in rows:
        for call in row.get("calls", []):
            keys = [call, call.rsplit(".", 1)[-1]]
            for callee_id in dict.fromkeys(cid for key in keys for cid in symbol_map.get(key, [])):
                key = (row["item_id"], callee_id, call)
                if callee_id != row["item_id"] and key not in seen:
                    seen.add(key)
                    edges.append({"repo": repo_name, "version": version, "caller_id": row["item_id"], "callee_id": callee_id, "call": call})
    return edges

In [ ]:
#| export
def reference_ingest(
    name: str | None = None,  # Registry name to ingest
    all: bool = False,  # Ingest all registered repositories
    path: str | None = None,  # Override reference home
    force: bool = False,  # Reingest even when resolved version appears unchanged
):
    "Clone registered references and index Python/notebook implementations."
    data = _load_reference_registry(path)
    if not data["repos"]: raise ValueError("No reference repositories registered")
    names = list(data["repos"]) if all else [name or next(iter(data["repos"]))]
    reports = []
    with tempfile.TemporaryDirectory(prefix="nbskill-reference-") as tmp:
        for repo_name in names:
            if repo_name not in data["repos"]: raise KeyError(f"Unknown reference repo: {repo_name}")
            repo = data["repos"][repo_name]
            checkout = Path(tmp) / repo_name
            resolved = _checkout_reference(repo, checkout)
            repo_kind = "nbdev" if _nbdev_config(checkout) else "python"
            if not force and repo.get("resolved_version") == resolved and repo.get("item_count") and repo.get("repo_kind") == repo_kind:
                reports.append({"name": repo_name, "resolved_version": resolved, "repo_kind": repo_kind, "skipped": True, "item_count": repo["item_count"]})
                continue
            package = _infer_package(checkout, repo)
            rows = _extract_reference_rows(checkout, repo, resolved, package)
            count = _write_reference_rows(repo_name, resolved, rows, path=path)
            repo.update({"package": package, "repo_kind": repo_kind, "resolved_version": resolved, "last_indexed_ts": time.time(), "item_count": count})
            reports.append({"name": repo_name, "resolved_version": resolved, "repo_kind": repo_kind, "package": package, "item_count": count})
    _write_reference_registry(data, path)
    sync = _sync_lancedb(path)
    return {"path": str(_lancedb_path(path)), "ingested": reports, "index": sync}

In [ ]:
#| export
def reference_list(path: str | None = None):
    "Return registered reference repositories and indexed item counts."
    data = _load_reference_registry(path)
    counts = {}
    for row in _lance_rows(path, "items"):
        counts[row.get("repo")] = counts.get(row.get("repo"), 0) + 1
    repos = []
    for name, repo in sorted(data["repos"].items()):
        repos.append({**repo, "indexed_items": counts.get(name, 0)})
    return {"path": str(_lancedb_path(path)), "count": len(repos), "repos": repos}

In [ ]:
#| export
def _repo_filter(repos):
    if repos is None: return None
    if isinstance(repos, str): return [item.strip() for item in repos.split(",") if item.strip()]
    return list(repos)

In [ ]:
#| export
def _query_tokens(query):
    return [token for token in re.findall(r"[A-Za-z0-9_]+", query.lower()) if len(token) > 1]

In [ ]:
#| export
def _reference_query_note():
    return "Reference queries use LanceDB vector and full-text search."

In [ ]:
#| export
def _current_dependencies(current_repo):
    pyproject = Path(current_repo or ".") / "pyproject.toml"
    if not pyproject.exists(): return set()
    text = pyproject.read_text(encoding="utf-8", errors="replace")
    names = set()
    for quoted in re.findall(r"['\"]([^'\"]+)['\"]", text):
        name = re.split(r"[<>=!~;\[]", quoted, 1)[0].strip()
        if name: names.add(_norm_dist_name(name))
    return names

In [ ]:
#| export
def _dependency_status(package, current_repo="."):
    if not package: return "unknown"
    normalized = _norm_dist_name(package)
    if normalized in _current_dependencies(current_repo): return "direct_import"
    import_name = package.replace("-", "_")
    try:
        if importlib.util.find_spec(import_name) is not None: return "direct_import"
    except (ImportError, ValueError): pass
    return "new_dependency"

In [ ]:
#| export
def _item_by_id(item_id, path=None):
    for row in _lance_rows(path, "items"):
        if row.get("item_id") == item_id: return row
    return None

In [ ]:
#| export
def _branch_for_item(item_id, path=None, limit=8):
    branch = []
    for edge in _lance_rows(path, "edges"):
        relation, other_id = None, None
        if edge.get("caller_id") == item_id:
            relation, other_id = "callee", edge.get("callee_id")
        elif edge.get("callee_id") == item_id:
            relation, other_id = "caller", edge.get("caller_id")
        if relation and (row := _item_by_id(other_id, path=path)):
            branch.append({**_public_reference_hit(row), "relation": relation, "call": edge.get("call")})
        if len(branch) >= limit: break
    return branch

In [ ]:
#| export
def _first_docstring_line(docstring):
    for line in str(docstring or "").strip().splitlines():
        text = line.strip()
        if text: return text
    return ""

def _public_reference_hit(row):
    tags = row.get("tags") or []
    if isinstance(tags, str):
        try: tags = json.loads(tags)
        except json.JSONDecodeError: tags = [tags]
    docstring = row.get("docstring") or ""
    return {
        "repo": row.get("repo"), "version": row.get("version"), "package": row.get("package") or None,
        "repo_kind": row.get("repo_kind") or None, "kind": row.get("kind"), "module": row.get("module"), "symbol": row.get("symbol"),
        "path": row.get("path"), "cell_id": row.get("cell_id") or None,
        "line": row.get("start_line"), "signature": row.get("signature"),
        "docstring": docstring, "docstring_first_line": _first_docstring_line(docstring),
        "source": row.get("source"), "tags": list(tags), "returned_count": int(row.get("returned_count") or 0),
        "score": row.get("_reference_score"), "search_sources": row.get("_search_sources") or [],
        "embedding_backend": row.get("embedding_backend"),
    }

In [ ]:
#| export
def _bump_reference_return_counts(rows, path=None):
    ids = {row.get("item_id") for row in rows if row.get("item_id")}
    if not ids: return {}
    updated, counts = [], {}
    for row in _lance_rows(path, "items"):
        row = dict(row)
        if row.get("item_id") in ids:
            row["returned_count"] = int(row.get("returned_count") or 0) + 1
            counts[row["item_id"]] = row["returned_count"]
        updated.append(row)
    if counts: _write_lance_table(path, "items", updated)
    return counts

def reference_query(
    query: str,  # Natural-language implementation query
    top_k: int = 3,  # Number of implementation hits to return
    include_branch: bool = False,  # Include direct same-repo callers and callees
    current_repo: str = ".",  # Current project for dependency status
    repos=None,  # Optional repo name or comma-separated names to search
    path: str | None = None,  # Override reference home
    kind: str | None = None,  # Optional item kind filter: readme, module, function, class, method
    package: str | None = None,  # Optional package filter
    module: str | None = None,  # Optional module filter
    symbol: str | None = None,  # Optional symbol filter
):
    "Search indexed reference implementations."
    repo_names = _repo_filter(repos)
    rows, reason, backend = _lancedb_search(query, top_k=top_k, repos=repo_names, path=path, kind=kind, package=package, module=module, symbol=symbol)
    counts = _bump_reference_return_counts(rows[:top_k], path=path)
    hits = []
    for row in rows[:top_k]:
        if row.get("item_id") in counts: row["returned_count"] = counts[row["item_id"]]
        hit = _public_reference_hit(row)
        hit["dependency_status"] = _dependency_status(hit.get("package"), current_repo=current_repo)
        if include_branch: hit["branch"] = _branch_for_item(row["item_id"], path=path)
        hits.append(hit)
    filters = {"repos": repo_names, "kind": kind, "package": package, "module": module, "symbol": symbol}
    return {"query": query, "backend": backend, "note": reason, "filters": filters, "count": len(hits), "hits": hits}

Small reference API examples:

In [ ]:
example_home = reference_home(demo_path("reference-example-home"))
assert example_home.name == "reference-example-home"
remove_demo_path(example_home)

In [ ]:
example_vector = _reference_vector("make page endpoint")
assert len(example_vector) > 100

Reference repositories are managed through the CLI-facing registry. The executable contract uses a local git repository so ingestion never depends on network access.

In [ ]:
from fastcore.nbio import write_nb as _write_raw_nb
from nbskill.foundation import demo_path, remove_demo_path

repo_root = demo_path("reference-src")
home = demo_path("reference-home")
project = demo_path("reference-project")
try:
    project.mkdir(parents=True, exist_ok=True)
    (repo_root / "demo_pkg").mkdir(parents=True)
    (repo_root / "demo_pkg" / "__init__.py").write_text("", encoding="utf-8")
    (repo_root / "demo_pkg" / "web.py").write_text(
        "def render_button(label):\n"
        "    return f'<button>{label}</button>'\n\n"
        "def fasthtml_route(app):\n"
        "    \"Build a FastHTML route with a reusable button.\"\n"
        "    return app.get('/')(lambda: render_button('Save'))\n\n"
        "class RouteBuilder:\n"
        "    def build(self, app):\n"
        "        return fasthtml_route(app)\n",
        encoding="utf-8",
    )
    (repo_root / "README.md").write_text("Example FastHTML website patterns.", encoding="utf-8")
    _write_raw_nb(new_nb([mk_cell("Notebook docs for a route.", cell_type="markdown"), mk_cell("def notebook_route(app):\n    return app")]), repo_root / "example.ipynb")
    subprocess.run(["git", "init"], cwd=repo_root, check=True, capture_output=True)
    subprocess.run(["git", "add", "."], cwd=repo_root, check=True, capture_output=True)
    subprocess.run(["git", "-c", "user.name=Nbskill", "-c", "user.email=nbskill@example.com", "commit", "-m", "init"], cwd=repo_root, check=True, capture_output=True)
    added = reference_add(str(repo_root), name="demo", package="demo_pkg", path=str(home))
    ingested = reference_ingest(name="demo", path=str(home), force=True)
    repos = _lance_rows(str(home), "repos")
    items = _lance_rows(str(home), "items")
    edges = _lance_rows(str(home), "edges")
    listed = reference_list(path=str(home))
    assert added["count"] == 1
    assert repos and repos[0]["name"] == "demo"
    assert ingested["ingested"][0]["item_count"] >= 5
    assert len(items) >= 5
    assert edges
    assert listed["repos"][0]["indexed_items"] == len(items)
    (project / "pyproject.toml").write_text("[project]\ndependencies = ['demo-pkg']\n", encoding="utf-8")
    result = reference_query("create a fasthtml page handler with button", top_k=3, include_branch=True, current_repo=str(project), path=str(home), kind="function")
    route_hit = next(hit for hit in result["hits"] if hit["symbol"] == "fasthtml_route")
    assert result["backend"] == "lancedb_hybrid_bm25_vector"
    assert result["filters"]["kind"] == "function"
    assert route_hit["dependency_status"] == "direct_import"
    assert route_hit["returned_count"] == 1
    assert route_hit["search_sources"]
    assert any(item["relation"] == "callee" for item in route_hit.get("branch", []))
    structured = _item_row(
        "patterns", "v1", "fasthtml", "function", "web.routes", "save_route", Path("patterns.py"),
        "def save_route(app):\n    return app.get('/save')(lambda: 'saved')",
        signature="def save_route(app)",
        docstring="Build a FastHTML route endpoint page handler.",
        imports=["fasthtml"],
    )
    _write_lance_table(str(home), "items", [*_lance_rows(str(home), "items"), structured])
    similar = reference_query("make page endpoint", top_k=3, repos=["patterns"], path=str(home), kind="function")
    assert similar["backend"] == "lancedb_hybrid_bm25_vector"
    assert similar["hits"][0]["symbol"] == "save_route"
    assert similar["hits"][0]["returned_count"] == 1
finally:
    remove_demo_path(repo_root)
    remove_demo_path(home)
    remove_demo_path(project)

### Retrieval Experiments

This small experiment compares four search surfaces on a tiny structured corpus:

- `tags_fts`: LanceDB full-text search over tags plus symbol/doc text, with no vector similarity.
- `docstring`: vector similarity over docstrings only.
- `signature_doc_return`: vector similarity over the signature, docstring, and return statements.
- `full_body`: vector similarity over the whole indexed source block.

The expected symbols mirror common fastcore lookups: `store_attr`, `patch`, and `compose`.

In [ ]:
def _experiment_returns(source):
    try: tree = ast.parse(source or "")
    except SyntaxError: return ""
    vals = []
    for node in ast.walk(tree):
        if isinstance(node, ast.Return) and node.value is not None:
            try: vals.append(ast.unparse(node.value))
            except Exception: pass
    return "\n".join(vals)

In [ ]:
def _experiment_text(row, mode):
    if mode == "docstring": return row.get("docstring", "")
    if mode == "signature_doc_return":
        return "\n".join([row.get("signature", ""), row.get("docstring", ""), _experiment_returns(row.get("source", ""))])
    if mode == "full_body": return row.get("source", "")
    return " ".join([*(row.get("tags") or []), row.get("symbol", ""), row.get("signature", ""), row.get("docstring", "")])

In [ ]:
def _experiment_table(home, rows, mode):
    table_rows = []
    for row in rows:
        text = _experiment_text(row, mode)
        table_rows.append({**row, "search_text": text, "vector": _reference_vector(text)})
    table = _reference_db(home).create_table(f"reference_experiment_{mode}", data=table_rows, mode="overwrite")
    try: table.create_fts_index("search_text", replace=True)
    except Exception: pass
    return table

In [ ]:
def _experiment_search(table, query, mode, top_k=3):
    if mode == "tags_fts":
        rows = table.search(query, query_type="fts").limit(top_k).to_list()
    else:
        rows = table.search(_reference_vector(query)).metric("cosine").limit(top_k).to_list()
    return [row["symbol"] for row in rows]

In [ ]:
experiment_home = demo_path("reference-experiment-home")
experiment_rows = [
    _item_row("mini", "v1", "mini", "function", "mini.basics", "store_attr", Path("mini.py"),
              "def store_attr(names=None, self=None, **attrs):\n    return _store_attr(self, **attrs)",
              signature="def store_attr(names=None, self=None, **attrs)",
              docstring="Store params named in comma-separated names from calling context into attrs in self."),
    _item_row("mini", "v1", "mini", "function", "mini.basics", "patch", Path("mini.py"),
              "def patch(f=None, *, cls_method=False):\n    return patch_to(cls_method=cls_method)(f)",
              signature="def patch(f=None, *, cls_method=False)",
              docstring="Decorator: add f to the first parameter's class based on type annotations."),
    _item_row("mini", "v1", "mini", "function", "mini.basics", "compose", Path("mini.py"),
              "def compose(*funcs):\n    def _inner(x):\n        for f in funcs: x = f(x)\n        return x\n    return _inner",
              signature="def compose(*funcs)",
              docstring="Create a function that composes all functions in funcs."),
    _item_row("mini", "v1", "mini", "function", "mini.foundation", "flatmap", Path("mini.py"),
              "def flatmap(f, xs):\n    return [y for x in xs for y in f(x)]",
              signature="def flatmap(f, xs)",
              docstring="Apply f to each element and flatten the results into a single list."),
]
experiment_queries = {
    "store constructor arguments as attributes": "store_attr",
    "patch a method onto a class": "patch",
    "compose functions into a pipeline": "compose",
}
experiment_results = {}
try:
    for mode in ("tags_fts", "docstring", "signature_doc_return", "full_body"):
        table = _experiment_table(str(experiment_home), experiment_rows, mode)
        experiment_results[mode] = {
            query: _experiment_search(table, query, mode, top_k=3)
            for query in experiment_queries
        }
finally:
    remove_demo_path(experiment_home)
experiment_results

In [ ]:
experiment_accuracy = {
    mode: sum(hits and hits[0] == expected for query, expected in experiment_queries.items() for hits in [mode_results[query]])
    for mode, mode_results in experiment_results.items()
}
assert experiment_accuracy["signature_doc_return"] >= experiment_accuracy["docstring"]
assert experiment_accuracy["signature_doc_return"] >= experiment_accuracy["full_body"]
experiment_accuracy

### How MCP loads the right memory

MCP read tools already know what the agent looked at: `nb_overview` has a notebook path, `nb_cell` has a notebook path and cell id, `show_doc` resolves a symbol to a cell, and edit tools know the cells they changed. The MCP response layer uses that scope to run `knowledge_style_problems` and then filters warnings down to those paths and cell ids.

That means the LLM does not need a separate retrieval step for normal work. When it reads a cell containing a stored pattern, the warning rides along with the tool result. `get_knowledge` is still useful when the agent wants to inspect or update the memory itself.